# Diacritics Restoration project
author: [Jakub Łabuz](https://github.com/jakseluz)

## Introduction

The project focuses on diacritics restoration in Polish language words taking the context into account.

e.g. Labuz -> Łabuz


### Research
Articles which I found to be adequate for the problem:
- „Diacritics Restoration Using Neural Networks”\
(Jakub N´aplava, Milan Straka, Pavel Straˇn´ak, Jan Hajiˇc, 2018)
- [„Diacritics Restoration using BERT with Analysis on Czech language”\
(Jakub N´aplava, Milan Straka, Jana Strakov´a, 2021)](https://arxiv.org/abs/2105.11408)
- [„Correcting Diacritics and Typos with a ByT5 Transformer Model”\
(Lukas Stankeviˇcius, Mantas Lukoˇseviˇcius, Jurgita Kapoˇci¯ut˙e-Dzikien˙e,
Monika Briedien˙e, Tomas Krilaviˇcius, 2022)](https://arxiv.org/abs/2201.13242)
- [„Dilated Convolutional Neural Networks for Lightweight Diacritics
Restoration”\
(B´alint Csan´ady, Andr´as Luk´acs, 2022)](https://arxiv.org/abs/2201.06757)
- [„Romanian Diacritics Restoration Using Recurrent Neural Networks”\
(Stefan Ruseti, Teodor-Mihai Cotet, and Mihai Dascalu, 2020)](https://arxiv.org/abs/2009.02743).


### Main possible approaches
- character-level classification
- transformers connected with an external LLM
- sequence-to-sequence.


### Project assumptions
- self-supervised learning
- batch generating during the learning process - by diacritics removal.


### Dataset I used
- Polish Wikipedia, using [datasets library](https://huggingface.co/docs/datasets/index) - large and fully sufficient for learning.
Wikimedia Wikipedia (PL):
[https://huggingface.co/datasets/wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia):
    ```python
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.pl")
    ```


### Other datasets - promising but not needed here:
- CulturaX (Polish subset):
https://huggingface.co/datasets/uonlp/CulturaX
- hand-annotated million NJKP corpus:
https://nkjp.pl/index.php?page=14lang=0
- CLARIN-PL corpuses:
https://clarin-pl.eu/catalog/resources - e.g. Parliamentary sessions of Sejm & Senat RP (300 milion of
tokens)
- PolEval (NLP competitions):
http://poleval.pl/ - e.g. to compare used data with competitors solutions.


### Metrics for evaluation
- ~~accuracy~~ - not especially helpful - can be good even when a model does not work (diacritics percentages in words are quite low)
- WER (Word Error Rate) - mistaken word percentage 
- CER (Character Error Rate) - mistaken character percentage
- DER (Diacritic Error Rate) - mistaken diacritics percentage.

From the above, I have chosen CER to be the most valuable indicator.
That is beacuse models can - not only restore diacritics where they are expected to do it - but also where the letter should be untouched.
CER takes into account both situations and present the general model efficiency when considering the project topic.

## Project code

In [1]:
%load_ext autoreload
%autoreload 2

### Import Pytorch and print the configuration information

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


#### Dataset 'preconfiguration tests' - if you want to check how the dataset looks like

In [ ]:
from diacritics_restoration.utils import get_wikipedia_data

lista = [
    text for text in get_wikipedia_data(num_articles=5).head()["text"].tolist()
]
print("Original texts:")
print(lista)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Original texts:
['Bodenmais – uzdrowiskowa gmina targowa w Niemczech, w kraju związkowym Bawaria, w rejencji Dolna Bawaria, w regionie Donau-Wald, w powiecie Regen. Leży w Lesie Bawarskim, około 10\xa0km na północ od miasta Regen, nad rzeką Inn, przy linii kolejowej Bodenmais – Zwiesel, ok. 10\xa0km od granicy państwa z Czechami.\n\nDemografia\n\nOświata \n(na 1999)\nW gminie znajduje się 75 miejsc przedszkolnych (89 dzieci) oraz szkoła podstawowa (21 nauczycieli, 320 uczniów).\n\nPowiat Regen\nUzdrowiska w Niemczech\nGminy w Bawarii', 'Enrique Hector Scalabroni (ur. 20 października 1949 w Alta Gracia) – argentyński inżynier i projektant wyścigowy.\n\nŻyciorys \nUkończył studia na Uniwersytecie Buenos Aires. W 1985 roku podjął pracę w Formule 1, w zespole Williams. Pełnił tam rolę asystenta ds. projektowania, będąc podwładnym Patricka Heada. W 1988 roku zaprojektował własny samochód Formuły 1 z dwoma kołami

In [ ]:
from diacritics_restoration.utils import (
    get_wikipedia_data,
    DiacriticsDataset,
    train_model,
)

import torch
from torch.utils.data import DataLoader
from torch import nn

/home/jakseluz/miniconda3/envs/mlnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### data preparation

In [ ]:
def get_dataset_and_dataloader(
    num_articles=10_000, batch_size=256, shuffle=False
) -> tuple[DiacriticsDataset, DataLoader]:
    dataset = DiacriticsDataset(
        get_wikipedia_data(num_articles=num_articles)["text"].tolist()
    )
    print("Dataset size:", len(dataset))
    return dataset, DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
dataset, dataloader = get_dataset_and_dataloader(
    num_articles=10000, batch_size=256, shuffle=True
)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Dataset size: 151609


### Dilated 1D CNN - first

In [ ]:
from diacritics_restoration.models import DiacriticsCNN

#### model definition

In [ ]:
model = DiacriticsCNN(vocab_size=dataset.processor.vocab_size)
criterion = nn.CrossEntropyLoss(ignore_index=dataset.processor.pad_token_id)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [7]:
from torchinfo import summary

summary(model, input_size=(256, 256), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
DiacriticsCNN                            [256, 105, 256]           --
├─Embedding: 1-1                         [256, 256, 128]           13,440
├─Conv1d: 1-2                            [256, 256, 256]           98,560
├─ModuleList: 1-3                        --                        --
│    └─ResidualDilatedBlock: 2-1         [256, 256, 256]           --
│    │    └─Conv1d: 3-1                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-2             [256, 256, 256]           512
│    │    └─Conv1d: 3-3                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-4             [256, 256, 256]           512
│    └─ResidualDilatedBlock: 2-2         [256, 256, 256]           --
│    │    └─Conv1d: 3-5                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-6             [256, 256, 256]           512
│    │    └─Conv1d: 3-7                  [256, 256, 256]   

#### training

In [ ]:
print("Starting training...")
train_model(
    model, dataloader, epochs=40, criterion=criterion, optimizer=optimizer
)

Starting training...


Epoch 1/40: 100%|██████████| 597/597 [02:22<00:00,  4.20it/s, loss=0.0095] 


Epoch 1/40 - Average Loss: 0.0471
Best model saved (models/DiacriticsCNN/DiacriticsCNN_1_0.04706514083391907_2026-04-29_18-10-56_best_model.pt) with loss: 0.04706514083391907


Epoch 2/40: 100%|██████████| 597/597 [02:22<00:00,  4.20it/s, loss=0.00694]


Epoch 2/40 - Average Loss: 0.0076
Best model saved (models/DiacriticsCNN/DiacriticsCNN_2_0.007594179504019032_2026-04-29_18-13-18_best_model.pt) with loss: 0.007594179504019032


...

Epoch 14/40 - Average Loss: 0.0012
Best model saved (models/DiacriticsCNN/DiacriticsCNN_14_0.0011721672543864292_2026-04-29_18-41-52_best_model.pt) with loss: 0.0011721672543864292


Epoch 15/40: 100%|██████████| 597/597 [02:23<00:00,  4.16it/s, loss=0.000863]


Epoch 15/40 - Average Loss: 0.0011
Best model saved (models/DiacriticsCNN/DiacriticsCNN_15_0.0010870353664291814_2026-04-29_18-44-16_best_model.pt) with loss: 0.0010870353664291814


Epoch 16/40:  17%|█▋        | 99/597 [00:24<02:01,  4.10it/s, loss=0.000815]


KeyboardInterrupt: 

#### model evaluation

In [ ]:
from diacritics_restoration.utils import DiacriticsRestorer
from diacritics_restoration.models import DiacriticsCNN

import torch

/home/jakseluz/miniconda3/envs/mlnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### the latest restorer

In [ ]:
import glob
import os
from diacritics_restoration.utils.processor import CharacterProcessor


def load_latest_restorer(path: str) -> DiacriticsRestorer:
    print("Loading best model weights...")
    model_files = glob.glob(path)
    latest_model_file = max(model_files, key=os.path.getctime)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = CharacterProcessor()
    model = DiacriticsCNN(vocab_size=processor.vocab_size)
    state = torch.load(latest_model_file, map_location=device)
    model.load_state_dict(state)
    model.to(device=device)
    model.eval()
    restorer = DiacriticsRestorer(
        model=model, processor=processor, device=device
    )
    return restorer

##### sample uses

In [ ]:
def test(
    restorer: DiacriticsRestorer,
    test_text: str = "To jest przykladowy tekst bez znakow diakrytycznych.",
) -> None:
    restored_text = restorer.restore(test_text)
    print("Original:", test_text)
    print("Restored:", restored_text)


restorer = load_latest_restorer(path="models/DiacriticsCNN/*best_model.pt")

test(restorer)
test(restorer, "Zazolc to gory, a na niej siedzi zielony zolw.")
test(restorer, "Wczoraj bylem w sklepie i kupilem mleko oraz chleb.")
test(restorer, "Czy moglbys mi powiedziec, gdzie jest najblizsza stacja metra?")

Loading best model weights...
Original: To jest przykladowy tekst bez znakow diakrytycznych.
Restored: To jest przykładowy tekst bez znaków diakrytycznych.
Original: Zazolc to gory, a na niej siedzi zielony zolw.
Restored: Zażolć to góry, a na niej siedzi zielony żółw.
Original: Wczoraj bylem w sklepie i kupilem mleko oraz chleb.
Restored: Wczoraj byłem w sklepie i kupiłem mleko oraz chleb.
Original: Czy moglbys mi powiedziec, gdzie jest najblizsza stacja metra?
Restored: Czy mógłbyś mi powiedzieć, gdzie jest najbliższą stacją metra?


##### **CER** - Character Error Rate (Dilated 1D CNN)

In [4]:
print(restorer.calculate_history_character_error_rate())

0.08056872037914692


##### Evaluate on random articles

In [ ]:
from diacritics_restoration.utils.testing import evaluate_restorer_on_articles

evaluate_restorer_on_articles(
    restorer=restorer, num_articles=1000, batch_size=256
)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Dataset size: 14359

Example 0
PRED: Melinda F<UNK>bi<UNK>n (ur. <UNK><UNK> czerwca <UNK><UNK><UNK><UNK> w Tatab<UNK>nyi) <UNK> węgierska zawodniczka MMA.<UNK><UNK>Sztuki walki zaczęła trenować w wieku <UNK><UNK> lat, zaczynając od karate sh<UNK>t<UNK>kan. W wieku <UNK><UNK> lat wyjechała wraz z matką do Sztokholmu, gdzie zaczęła trenować muay thai, a po powrocie
TRUE: Melinda F<UNK>bi<UNK>n (ur. <UNK><UNK> czerwca <UNK><UNK><UNK><UNK> w Tatab<UNK>nyi) <UNK> węgierska zawodniczka MMA.<UNK><UNK>Sztuki walki zaczęła trenować w wieku <UNK><UNK> lat, zaczynając od karate sh<UNK>t<UNK>kan. W wieku <UNK><UNK> lat wyjechała wraz z matką do Sztokholmu, gdzie zaczęła trenować muay thai, a po powrocie
Differences: 0 / 328

Example 1
PRED: trenować w wieku <UNK><UNK> lat, zaczynając od karate sh<UNK>t<UNK>kan. W wieku <UNK><UNK> lat wyjechała wraz z matką do Sztokholmu, gdzie zaczęła trenować muay thai, a po powrocie 